<a href="https://colab.research.google.com/github/yogesh-bhattarai/Django/blob/main/lab3_class_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

In [2]:
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

Spark UI available at: http://f8a2ad423b4d:4040


In [3]:
spark.sparkContext.setLogLevel("INFO")

In [4]:
import psutil
print(f"CPU Usage: {psutil.cpu_percent()}%")
print(f"Memory Usage: {psutil.virtual_memory().percent}%")

CPU Usage: 36.0%
Memory Usage: 11.4%


In [10]:
# Mount Gdrive
from google.colab import drive
drive.mount('/content/drive')
# Load the data from a CSV file
df = spark.read.csv("/content/drive/MyDrive/dataset/Diabetes.csv", header=True, inferSchema=True)

# get familiar with data
df.show()

# more info
print("Total Records",df.count())
print("Total Partitions ",df.rdd.getNumPartitions())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
| id|age|   sex|  bmi|    bp|    tc|  ldl|  hdl| tch|  ltg|   glu|progression|
+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
|  0| 22|  Male|20.33|106.25|140.89|-0.45|62.83|0.77|-0.54|130.98|      29.83|
|  1| 41|Female|25.99|132.23|129.64|-1.11|37.26|0.81| 1.64|151.19|      46.66|
|  2| 51|Female|32.76| 127.0|220.36|-1.69|49.56|0.41|-0.88|176.22|      59.97|
|  3| 26|  Male|35.87| 138.4|194.19|-0.04|55.57|0.45|-1.38|125.32|      42.44|
|  4| 42|Female| 21.5|122.33|275.79| 1.19|63.64|0.54|-0.69|184.72|      49.36|
|  5| 47|  Male|31.62|137.18|232.35|-1.65|36.68|0.26| 1.63| 99.83|      54.15|
|  6| 22|  Male|37.06|106.47|244.34|-0.07|34.22|0.48|-1.72| 65.41|      39.55|
|  7| 23|  Male|26.75|129.39|177.69|-0.37|42.03|0.68| 0.82|180.71|      33.12|
| 

In [11]:
# show Schema,Prints the structure of the dataset
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- bp: double (nullable = true)
 |-- tc: double (nullable = true)
 |-- ldl: double (nullable = true)
 |-- hdl: double (nullable = true)
 |-- tch: double (nullable = true)
 |-- ltg: double (nullable = true)
 |-- glu: double (nullable = true)
 |-- progression: double (nullable = true)



In [12]:
#Statistical Analysis
df.describe().show()

+-------+-----------------+------------------+-------+------------------+------------------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-----------------+
|summary|               id|               age|    sex|               bmi|                bp|                tc|                 ldl|              hdl|                tch|                 ltg|               glu|      progression|
+-------+-----------------+------------------+-------+------------------+------------------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-----------------+
|  count|          3045570|           3045570|3045570|           3045570|           3045570|           3045570|             3045570|          3045570|            3045570|             3045570|           3045570|          3045570|
|   mean|        1522784.5| 39.50258309610352|   NULL|30.005921735504103|120.0061583

In [13]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 25.3%
Memory Usage after csv file: 15.4%


In [22]:
# check missing or null values for each column
from pyspark.sql.functions import col,isnan, isnull, when, count
df.select([count(when(isnull(c), c))
.alias(c) for c in df.columns]).show()

+---+---+---+---+---+---+---+---+---+---+-----------+------+
| id|age|bmi| bp| tc|ldl|hdl|tch|ltg|glu|progression|gender|
+---+---+---+---+---+---+---+---+---+---+-----------+------+
|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|          0|     0|
+---+---+---+---+---+---+---+---+---+---+-----------+------+



In [17]:
# convert categorical column into numbers
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol = 'sex', outputCol = 'gender')
df = indexer.fit(df).transform(df)
#df.show(30)

In [18]:
df.show()

+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+------+
| id|age|   sex|  bmi|    bp|    tc|  ldl|  hdl| tch|  ltg|   glu|progression|gender|
+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+------+
|  0| 22|  Male|20.33|106.25|140.89|-0.45|62.83|0.77|-0.54|130.98|      29.83|   1.0|
|  1| 41|Female|25.99|132.23|129.64|-1.11|37.26|0.81| 1.64|151.19|      46.66|   0.0|
|  2| 51|Female|32.76| 127.0|220.36|-1.69|49.56|0.41|-0.88|176.22|      59.97|   0.0|
|  3| 26|  Male|35.87| 138.4|194.19|-0.04|55.57|0.45|-1.38|125.32|      42.44|   1.0|
|  4| 42|Female| 21.5|122.33|275.79| 1.19|63.64|0.54|-0.69|184.72|      49.36|   0.0|
|  5| 47|  Male|31.62|137.18|232.35|-1.65|36.68|0.26| 1.63| 99.83|      54.15|   1.0|
|  6| 22|  Male|37.06|106.47|244.34|-0.07|34.22|0.48|-1.72| 65.41|      39.55|   1.0|
|  7| 23|  Male|26.75|129.39|177.69|-0.37|42.03|0.68| 0.82|180.71|      33.12|   1.0|
|  8| 44|Female|28.38|125.23|276.99| 0.69|55.18| 1.1|-

In [20]:
df= df.drop('sex')

In [21]:
# check missing or null values for each column
from pyspark.sql.functions import col,isnan, when, count
df.select([count(when(isnan(c) | col(c).isNull(), c))
.alias(c) for c in df.columns]).show()

+---+---+---+---+---+---+---+---+---+---+-----------+------+
| id|age|bmi| bp| tc|ldl|hdl|tch|ltg|glu|progression|gender|
+---+---+---+---+---+---+---+---+---+---+-----------+------+
|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|          0|     0|
+---+---+---+---+---+---+---+---+---+---+-----------+------+



In [23]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols = ["age", "gender", "bmi","bp","tc","ldl","hdl","tch","ltg","glu"],
                           outputCol = "Features")

In [24]:
#StandardScaler
scaler = StandardScaler(inputCol = "Features",
                        outputCol = "scaled_Features")

In [25]:
#create linear regression model.
regressor = LinearRegression(labelCol = 'progression',
                             featuresCol = 'scaled_Features'
                             )

In [26]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 26.0%
Memory Usage after csv file: 17.0%


In [33]:
from pyspark.ml import Pipeline
pipeline  = Pipeline(stages = [assembler,scaler,regressor])
#--Saving the Pipeline
pipeline.write().overwrite().save("pipeline_LRsaved_model")

In [28]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 23.9%
Memory Usage after csv file: 16.9%


In [29]:
pipelineModel = Pipeline.load('./pipeline_LRsaved_model')

In [30]:
data_train , data_test = df.randomSplit([0.7,0.3], seed = 123)

In [31]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 15.7%
Memory Usage after csv file: 16.9%


In [34]:
Model = pipeline.fit(data_train)

In [36]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 29.3%
Memory Usage after csv file: 25.9%


In [37]:
print("total LR cofficents",len(Model.stages[2].coefficients) )
print("Cofficientents",Model.stages[2].coefficients)
print("Intecept",Model.stages[2].intercept)

total LR cofficents 10
Cofficientents [8.502430305819576,-5.100187647371064e-06,3.4981838381075034,0.09996892814261281,-1.7903152557385793e-06,1.0011710635500897,-6.082063603222137e-07,-0.019992637345279157,-0.8003327984739753,-3.4325509191450796e-07]
Intecept 7.636572602858762e-05


In [38]:
#n the prediction phase, we test our model on some unseen data.
pred = Model.transform(data_test)
pred.select('prediction', 'progression').show(10, truncate = False)

+------------------+-----------+
|prediction        |progression|
+------------------+-----------+
|59.97303581230069 |59.97      |
|39.552725911208285|39.55      |
|51.805025944731   |51.81      |
|60.75132257410312 |60.75      |
|41.111943395192114|41.11      |
|52.2342223825314  |52.23      |
|47.07732942834482 |47.08      |
|29.53012488000392 |29.53      |
|58.81243457194682 |58.81      |
|55.617820885904905|55.62      |
+------------------+-----------+
only showing top 10 rows


In [39]:
#create linear regression model.
Lasoregressor = LinearRegression(labelCol = 'progression',
                             featuresCol = 'scaled_Features',
                             elasticNetParam=1,
                             regParam=0.1
                             )
Lasaopipeline  = Pipeline(stages = [assembler,scaler,Lasoregressor])
LassoModel = Lasaopipeline.fit(data_train)

In [40]:
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 68.3%
Memory Usage after csv file: 26.3%


In [41]:
#n the prediction phase, we test our model on some unseen data.
lassopred = LassoModel.transform(data_test)
lassopred.select('prediction', 'progression').show(10, truncate = False)

+------------------+-----------+
|prediction        |progression|
+------------------+-----------+
|59.820248005071825|59.97      |
|39.59501025037714 |39.55      |
|51.52701310177945 |51.81      |
|60.64909540434239 |60.75      |
|41.54453598645243 |41.11      |
|52.38059086977667 |52.23      |
|47.09298458693716 |47.08      |
|29.482487288220497|29.53      |
|58.495734842361855|58.81      |
|55.56966410771528 |55.62      |
+------------------+-----------+
only showing top 10 rows


In [42]:
#create linear regression model.
Ridgeregressor = LinearRegression(labelCol = 'progression',
                             featuresCol = 'scaled_Features',
                             elasticNetParam=0,
                             regParam=0.1
                             )
Ridgepipeline  = Pipeline(stages = [assembler,scaler,Ridgeregressor])
RidgeModel = Ridgepipeline.fit(data_train)

In [ ]:
#n the prediction phase, we test our model on some unseen data.
Ridgepred = RidgeModel.transform(data_test)
Ridgepred.select('prediction', 'progression').show(10, truncate = False)

In [43]:
#Model Evaluation Spark Provides evaluation metrics
#for regression and classification tasks.
from pyspark.ml.evaluation import RegressionEvaluator
evaluator_mse = RegressionEvaluator(labelCol =
                                    'progression',
                                    predictionCol =
                                    'prediction',
                                    metricName =
                                    'mse')
# calculate MSE
mse1 = evaluator_mse.evaluate(pred)
mselasso = evaluator_mse.evaluate(lassopred)
mseridge = evaluator_mse.evaluate(Ridgepred)

evaluator_rmse = RegressionEvaluator(labelCol =
                                     'progression',
                                     predictionCol =
                                     'prediction',
                                     metricName =
                                     'rmse')
# calculate RMSE
rmse1 = evaluator_rmse.evaluate(pred)
rmse2_lasso = evaluator_rmse.evaluate(lassopred)
rmse3Ridge = evaluator_rmse.evaluate(Ridgepred)

evaluator_r2 = RegressionEvaluator(labelCol = 'progression',
                                   predictionCol = 'prediction',
                                   metricName = 'r2')
# calculate R_squared
r2_score1 = evaluator_r2.evaluate(pred)
r2_lasso = evaluator_r2.evaluate(lassopred)
r2_ridge = evaluator_r2.evaluate(Ridgepred)
# print the evaluation metrics
print('Regression - MSE: ', mse1, ', RMSE: ', rmse1, ', R^2: ', r2_score1)
print('Lasso - MSE: ', mselasso, ', RMSE: ', rmse2_lasso, ', R^2: ', r2_lasso)
print('Ridge - MSE: ', mseridge, ', RMSE: ', rmse3Ridge, ', R^2: ', r2_ridge)

NameError: name 'Ridgepred' is not defined

In [35]:
# plot
import matplotlib.pyplot as plt
import numpy as np

mse = [mse1, mselasso, mseridge]
rmse = [rmse1, rmse2_lasso, rmse3Ridge]
r2_score = [r2_score1, rmse2_lasso, r2_ridge]

positions = np.arange(len(mse))
bar_width = 0.2

plt.bar(positions - bar_width, mse, width = bar_width, label = 'MSE')
plt.bar(positions, rmse, width = bar_width, label = 'RMSE')
plt.bar(positions + bar_width, r2_score, width = bar_width, label = 'R2_Score')

NameError: name 'mse1' is not defined